# 第三周：用户行为分析与商业价值挖掘

**数据周期**：2017-11-01 ~ 2017-12-14  
**分析视角**：拉新 → 留存 → 活跃 → 转化 → 付费

本 Notebook 汇总第三周全部分析内容：
1. 数据加载与探索
2. 流量漏斗与核心 KPI
3. 留存分析 & DAU 趋势
4. RFM 用户分层与运营策略
5. 品类分析与用户价值报告
6. 交互式 HTML 看板

> 原始数据不含商品价格，GMV/ARPU/ARPPU 基于假设客单价 ¥50 估算。

In [ ]:
%matplotlib inline

import json
from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display

plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei", "PingFang SC", "Heiti SC"]
plt.rcParams["axes.unicode_minus"] = False

BASE_DIR = Path(".").resolve()
DATA_PATH = BASE_DIR / "data2.csv"
OUTPUT_DIR = BASE_DIR / "output"
CHARTS_DIR = OUTPUT_DIR / "charts"

ASSUMED_AOV = 50.0
REFERENCE_DATE = pd.Timestamp("2017-12-14")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"工作目录: {BASE_DIR}")

## 工具函数定义

In [ ]:
def load_data() -> pd.DataFrame:
    df = pd.read_csv(DATA_PATH, sep="\t")
    df["datetime"] = pd.to_datetime(df["time"], unit="s")
    df = df[(df["datetime"] >= "2017-11-01") & (df["datetime"] <= "2018-01-01")].copy()
    df["date"] = df["datetime"].dt.date
    return df


def score_r(series: pd.Series) -> pd.Series:
    """最近活跃天数：越近分数越高"""
    ranks = series.rank(method="first", ascending=True)
    return pd.qcut(ranks, 5, labels=[1, 2, 3, 4, 5]).astype(int)


def score_fm_purchase(counts: pd.Series) -> pd.Series:
    """购买频次/金额打分：未购买用户统一为 1 分，付费用户按分位打分"""
    scores = pd.Series(1, index=counts.index, dtype=int)
    buyers = counts[counts > 0]
    if len(buyers) == 0:
        return scores
    ranks = buyers.rank(method="first", ascending=False)
    buyer_scores = pd.qcut(ranks, 5, labels=[1, 2, 3, 4, 5]).astype(int)
    scores.loc[buyers.index] = buyer_scores
    return scores


def segment_rfm(r: int, f: int, m: int, bought: bool) -> str:
    if not bought:
        if r >= 4:
            return "高意向待转化用户"
        if r >= 2:
            return "活跃未付费用户"
        return "沉睡用户"
    r_high, f_high, m_high = r >= 4, f >= 4, m >= 4
    if r_high and f_high and m_high:
        return "重要价值用户"
    if r_high and not f_high and m_high:
        return "重要发展用户"
    if not r_high and f_high and m_high:
        return "重要保持用户"
    if not r_high and not f_high and m_high:
        return "重要挽留用户"
    if r_high and f_high and not m_high:
        return "一般价值用户"
    if r_high and not f_high and not m_high:
        return "一般发展用户"
    if not r_high and f_high and not m_high:
        return "一般保持用户"
    return "一般挽留用户"


def compute_funnel_metrics(df: pd.DataFrame) -> dict:
    pv = (df["behavior"] == "pv").sum()
    cart = (df["behavior"] == "cart").sum()
    fav = (df["behavior"] == "fav").sum()
    buy = (df["behavior"] == "buy").sum()
    uv = df["user_id"].nunique()
    paying_uv = df.loc[df["behavior"] == "buy", "user_id"].nunique()

    dau_series = df.groupby("date")["user_id"].nunique()
    dau_mean = float(dau_series.mean())
    dau_max = int(dau_series.max())

    # 新用户：首次行为落在当日
    first_seen = df.groupby("user_id")["datetime"].min().dt.date
    new_users_daily = first_seen.value_counts().sort_index()

    gmv = buy * ASSUMED_AOV
    arpu = gmv / uv
    arppu = gmv / paying_uv if paying_uv else 0

    return {
        "pv": int(pv),
        "uv": int(uv),
        "cart": int(cart),
        "fav": int(fav),
        "buy": int(buy),
        "paying_uv": int(paying_uv),
        "ctr_pv_to_cart": cart / pv if pv else 0,
        "ctr_pv_to_intent": (cart + fav) / pv if pv else 0,
        "cvr_pv_to_buy": buy / pv if pv else 0,
        "cvr_uv_to_buy": paying_uv / uv if uv else 0,
        "cart_to_buy_cvr": buy / cart if cart else 0,
        "gmv": gmv,
        "dau_mean": dau_mean,
        "dau_max": dau_max,
        "arpu": arpu,
        "arppu": arppu,
        "pay_rate": paying_uv / uv if uv else 0,
        "dau_series": {str(k): int(v) for k, v in dau_series.items()},
        "new_users_daily": {str(k): int(v) for k, v in new_users_daily.items()},
        "funnel_stages": {
            "浏览(PV)": int(pv),
            "加购": int(cart),
            "收藏": int(fav),
            "购买": int(buy),
        },
        "funnel_uv": {
            "活跃UV": int(uv),
            "有加购行为UV": int(df.loc[df["behavior"] == "cart", "user_id"].nunique()),
            "有收藏行为UV": int(df.loc[df["behavior"] == "fav", "user_id"].nunique()),
            "付费UV": int(paying_uv),
        },
    }


def compute_retention(df: pd.DataFrame) -> dict:
    user_first = df.groupby("user_id")["date"].min()
    active_by_date: dict = {}
    for d, grp in df.groupby("date"):
        active_by_date[d] = set(grp["user_id"].unique())

    dates = sorted(active_by_date.keys())
    d1_rates, d7_rates = [], []
    for d in dates:
        cohort = {u for u, fd in user_first.items() if fd == d}
        if not cohort:
            continue
        d1 = d + timedelta(days=1)
        d7 = d + timedelta(days=7)
        if d1 in active_by_date:
            d1_rates.append(len(cohort & active_by_date[d1]) / len(cohort))
        if d7 in active_by_date:
            d7_rates.append(len(cohort & active_by_date[d7]) / len(cohort))

    return {
        "d1_retention_avg": float(np.mean(d1_rates)) if d1_rates else 0,
        "d7_retention_avg": float(np.mean(d7_rates)) if d7_rates else 0,
    }


def build_rfm(df: pd.DataFrame) -> pd.DataFrame:
    buy_df = df[df["behavior"] == "buy"].copy()
    beh_counts = df.groupby(["user_id", "behavior"]).size().unstack(fill_value=0)
    all_activity = df.groupby("user_id").agg(
        last_active=("datetime", "max"),
        total_actions=("behavior", "count"),
    )
    buy_stats = buy_df.groupby("user_id").agg(
        last_buy=("datetime", "max"),
        buy_count=("behavior", "count"),
    )
    rfm = all_activity.join(buy_stats, how="left").join(beh_counts, how="left")
    rfm["buy_count"] = rfm["buy_count"].fillna(0)
    for col in ["pv", "cart", "fav", "buy"]:
        if col in rfm.columns:
            rfm[col] = rfm[col].fillna(0)
    rfm["recency_days"] = (REFERENCE_DATE - rfm["last_active"]).dt.days
    rfm["monetary"] = rfm["buy_count"] * ASSUMED_AOV
    rfm["bought"] = rfm["buy_count"] > 0
    rfm["R"] = score_r(rfm["recency_days"])
    rfm["F"] = score_fm_purchase(rfm["buy_count"])
    rfm["M"] = score_fm_purchase(rfm["monetary"])
    rfm["segment"] = rfm.apply(
        lambda row: segment_rfm(row["R"], row["F"], row["M"], row["bought"]), axis=1
    )
    # 高意向：有加购或收藏但未购买的近期用户
    intent_mask = (~rfm["bought"]) & ((rfm.get("cart", 0) > 0) | (rfm.get("fav", 0) > 0)) & (rfm["R"] >= 3)
    rfm.loc[intent_mask, "segment"] = "高意向待转化用户"
    rfm = rfm.reset_index()
    return rfm


def category_analysis(df: pd.DataFrame) -> pd.DataFrame:
    cat = df.groupby("category").agg(
        pv=("behavior", lambda x: (x == "pv").sum()),
        cart=("behavior", lambda x: (x == "cart").sum()),
        fav=("behavior", lambda x: (x == "fav").sum()),
        buy=("behavior", lambda x: (x == "buy").sum()),
        uv=("user_id", "nunique"),
    ).reset_index()
    cat["cvr"] = cat["buy"] / cat["pv"].replace(0, np.nan)
    cat["cart_rate"] = cat["cart"] / cat["pv"].replace(0, np.nan)
    return cat.sort_values("buy", ascending=False)


def save_charts(df: pd.DataFrame, metrics: dict, rfm: pd.DataFrame, cat_top: pd.DataFrame) -> None:
    CHARTS_DIR.mkdir(parents=True, exist_ok=True)

    # 漏斗图
    stages = ["浏览(PV)", "加购", "收藏", "购买"]
    values = [metrics["pv"], metrics["cart"], metrics["fav"], metrics["buy"]]
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ["#4C6EF5", "#228BE6", "#15AABF", "#12B886"]
    bars = ax.barh(stages[::-1], values[::-1], color=colors[::-1])
    for bar, val in zip(bars, values[::-1]):
        ax.text(bar.get_width() + max(values) * 0.01, bar.get_y() + bar.get_height() / 2,
                f"{val:,}", va="center", fontsize=10)
    ax.set_xlabel("行为次数")
    ax.set_title("用户行为转化漏斗")
    fig.tight_layout()
    fig.savefig(CHARTS_DIR / "funnel.png", dpi=150)
    plt.close(fig)

    # DAU 趋势
    dau = pd.Series(metrics["dau_series"])
    dau.index = pd.to_datetime(dau.index)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(dau.index, dau.values, marker="o", color="#4C6EF5", linewidth=2)
    ax.fill_between(dau.index, dau.values, alpha=0.15, color="#4C6EF5")
    ax.set_title("DAU 日活跃趋势")
    ax.set_ylabel("DAU")
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    fig.savefig(CHARTS_DIR / "dau_trend.png", dpi=150)
    plt.close(fig)

    # RFM 分层
    seg_counts = rfm["segment"].value_counts()
    fig, ax = plt.subplots(figsize=(9, 5))
    seg_counts.plot(kind="bar", ax=ax, color="#7950F2")
    ax.set_title("RFM 用户分层分布")
    ax.set_ylabel("用户数")
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    fig.savefig(CHARTS_DIR / "rfm_segments.png", dpi=150)
    plt.close(fig)

    # 品类 TOP10 转化
    top10 = cat_top.head(10)
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(top10))
    w = 0.35
    ax.bar(x - w / 2, top10["pv"], w, label="PV", color="#74C0FC")
    ax.bar(x + w / 2, top10["buy"], w, label="购买", color="#12B886")
    ax.set_xticks(x)
    ax.set_xticklabels(top10["category"].astype(str), rotation=45, ha="right")
    ax.set_title("TOP10 品类流量与购买对比")
    ax.legend()
    fig.tight_layout()
    fig.savefig(CHARTS_DIR / "category_top10.png", dpi=150)
    plt.close(fig)


def build_html_dashboard(metrics: dict, retention: dict) -> str:
    funnel = metrics["funnel_stages"]
    funnel_uv = metrics["funnel_uv"]
    dau_labels = list(metrics["dau_series"].keys())
    dau_values = list(metrics["dau_series"].values())
    new_labels = list(metrics["new_users_daily"].keys())
    new_values = list(metrics["new_users_daily"].values())

    def pct(a, b):
        return f"{a / b * 100:.2f}%" if b else "0%"

    return f"""<!DOCTYPE html>
<html lang="zh-CN">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>用户行为流量漏斗分析看板</title>
  <script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.1/dist/chart.umd.min.js"></script>
  <style>
    * {{ box-sizing: border-box; margin: 0; padding: 0; }}
    body {{ font-family: -apple-system, BlinkMacSystemFont, "PingFang SC", "Microsoft YaHei", sans-serif;
            background: #0f172a; color: #e2e8f0; padding: 24px; }}
    h1 {{ font-size: 1.75rem; margin-bottom: 8px; }}
    .subtitle {{ color: #94a3b8; margin-bottom: 24px; }}
    .grid {{ display: grid; gap: 16px; }}
    .kpi-grid {{ grid-template-columns: repeat(auto-fit, minmax(160px, 1fr)); margin-bottom: 24px; }}
    .card {{ background: #1e293b; border-radius: 12px; padding: 20px; border: 1px solid #334155; }}
    .kpi-card {{ text-align: center; }}
    .kpi-label {{ font-size: 0.85rem; color: #94a3b8; }}
    .kpi-value {{ font-size: 1.6rem; font-weight: 700; color: #38bdf8; margin-top: 6px; }}
    .kpi-sub {{ font-size: 0.75rem; color: #64748b; margin-top: 4px; }}
    .section-title {{ font-size: 1.1rem; margin-bottom: 16px; color: #f1f5f9; }}
    .two-col {{ grid-template-columns: 1fr 1fr; }}
    @media (max-width: 900px) {{ .two-col {{ grid-template-columns: 1fr; }} }}
    table {{ width: 100%; border-collapse: collapse; font-size: 0.9rem; }}
    th, td {{ padding: 10px 12px; text-align: left; border-bottom: 1px solid #334155; }}
    th {{ color: #94a3b8; font-weight: 500; }}
    .tag {{ display: inline-block; padding: 2px 8px; border-radius: 4px; font-size: 0.75rem;
            background: #334155; color: #a5b4fc; }}
    .funnel-step {{ display: flex; align-items: center; margin: 8px 0; }}
    .funnel-bar {{ height: 36px; border-radius: 6px; display: flex; align-items: center;
                   padding: 0 12px; font-weight: 600; font-size: 0.85rem; color: #fff; }}
  </style>
</head>
<body>
  <h1>用户行为流量漏斗分析看板</h1>
  <p class="subtitle">数据周期：2017-11-01 ~ 2017-12-14 | 平台视角：拉新 → 留存 → 活跃 → 转化 → 付费</p>

  <div class="grid kpi-grid">
    <div class="card kpi-card"><div class="kpi-label">PV 页面浏览量</div><div class="kpi-value">{metrics['pv']:,}</div></div>
    <div class="card kpi-card"><div class="kpi-label">UV 独立访客</div><div class="kpi-value">{metrics['uv']:,}</div></div>
    <div class="card kpi-card"><div class="kpi-label">DAU 日均活跃</div><div class="kpi-value">{metrics['dau_mean']:,.0f}</div><div class="kpi-sub">峰值 {metrics['dau_max']:,}</div></div>
    <div class="card kpi-card"><div class="kpi-label">CTR 意向率</div><div class="kpi-value">{metrics['ctr_pv_to_intent']:.2%}</div><div class="kpi-sub">(加购+收藏)/PV</div></div>
    <div class="card kpi-card"><div class="kpi-label">CVR 购买转化率</div><div class="kpi-value">{metrics['cvr_pv_to_buy']:.2%}</div><div class="kpi-sub">购买/PV</div></div>
    <div class="card kpi-card"><div class="kpi-label">GMV 成交总额</div><div class="kpi-value">¥{metrics['gmv']:,.0f}</div><div class="kpi-sub">假设客单价 ¥{ASSUMED_AOV:.0f}</div></div>
    <div class="card kpi-card"><div class="kpi-label">付费率</div><div class="kpi-value">{metrics['pay_rate']:.2%}</div><div class="kpi-sub">付费UV/总UV</div></div>
    <div class="card kpi-card"><div class="kpi-label">ARPPU</div><div class="kpi-value">¥{metrics['arppu']:.1f}</div><div class="kpi-sub">ARPU ¥{metrics['arpu']:.2f}</div></div>
  </div>

  <div class="grid two-col" style="margin-bottom:24px">
    <div class="card">
      <div class="section-title">行为转化漏斗（次数）</div>
      <div class="funnel-step"><div class="funnel-bar" style="width:100%;background:#4C6EF5">浏览 {funnel['浏览(PV)']:,}</div></div>
      <div class="funnel-step"><div class="funnel-bar" style="width:{funnel['加购']/funnel['浏览(PV)']*100:.1f}%;background:#228BE6;min-width:80px">加购 {funnel['加购']:,} ({pct(funnel['加购'], funnel['浏览(PV)'])})</div></div>
      <div class="funnel-step"><div class="funnel-bar" style="width:{max(funnel['收藏']/funnel['浏览(PV)']*100, 3):.1f}%;background:#15AABF;min-width:80px">收藏 {funnel['收藏']:,} ({pct(funnel['收藏'], funnel['浏览(PV)'])})</div></div>
      <div class="funnel-step"><div class="funnel-bar" style="width:{max(funnel['购买']/funnel['浏览(PV)']*100, 3):.1f}%;background:#12B886;min-width:80px">购买 {funnel['购买']:,} ({pct(funnel['购买'], funnel['浏览(PV)'])})</div></div>
      <p style="margin-top:12px;font-size:0.85rem;color:#94a3b8">加购→购买转化率：{metrics['cart_to_buy_cvr']:.2%}</p>
    </div>
    <div class="card">
      <div class="section-title">用户漏斗（UV）</div>
      <table>
        <tr><th>阶段</th><th>用户数</th><th>阶段转化率</th></tr>
        <tr><td>活跃 UV</td><td>{funnel_uv['活跃UV']:,}</td><td>—</td></tr>
        <tr><td>有加购行为</td><td>{funnel_uv['有加购行为UV']:,}</td><td>{pct(funnel_uv['有加购行为UV'], funnel_uv['活跃UV'])}</td></tr>
        <tr><td>有收藏行为</td><td>{funnel_uv['有收藏行为UV']:,}</td><td>{pct(funnel_uv['有收藏行为UV'], funnel_uv['活跃UV'])}</td></tr>
        <tr><td>付费 UV</td><td>{funnel_uv['付费UV']:,}</td><td><span class="tag">{pct(funnel_uv['付费UV'], funnel_uv['活跃UV'])}</span></td></tr>
      </table>
      <p style="margin-top:12px;font-size:0.85rem;color:#94a3b8">
        D1 留存率均值 {retention['d1_retention_avg']:.2%} | D7 留存率均值 {retention['d7_retention_avg']:.2%}
      </p>
    </div>
  </div>

  <div class="grid two-col">
    <div class="card"><div class="section-title">DAU 趋势</div><canvas id="dauChart" height="200"></canvas></div>
    <div class="card"><div class="section-title">每日新用户（拉新）</div><canvas id="newUserChart" height="200"></canvas></div>
  </div>

  <div class="card" style="margin-top:24px">
    <div class="section-title">平台盈利模式与术语说明</div>
    <table>
      <tr><th>盈利模式</th><th>说明</th><th>本数据映射</th></tr>
      <tr><td>交易佣金</td><td>平台从商家每笔订单抽取佣金（外卖约15-20%）</td><td>购买行为 × 佣金率</td></tr>
      <tr><td>广告收入</td><td>商家竞价排名、品牌专区、信息流广告</td><td>高 PV 低转化品类适合广告投放</td></tr>
      <tr><td>会员订阅</td><td>月卡/年卡，提升用户 LTV 与复购</td><td>高 RFM 分用户是会员转化目标</td></tr>
      <tr><td>配送/增值服务</td><td>配送费、保险、金融等</td><td>提升客单价与 ARPPU</td></tr>
    </table>
    <table style="margin-top:16px">
      <tr><th>术语</th><th>公式</th><th>业务含义</th></tr>
      <tr><td>PV</td><td>页面浏览次数</td><td>流量规模，反映曝光量</td></tr>
      <tr><td>UV</td><td>独立访客数</td><td>触达用户广度</td></tr>
      <tr><td>DAU</td><td>日活跃用户数</td><td>用户粘性核心指标</td></tr>
      <tr><td>CTR</td><td>点击/意向 ÷ 曝光</td><td>内容或商品吸引力</td></tr>
      <tr><td>CVR</td><td>转化 ÷ 流量</td><td>漏斗效率</td></tr>
      <tr><td>GMV</td><td>成交总额</td><td>平台交易规模</td></tr>
      <tr><td>ARPU</td><td>总收入 ÷ 总用户</td><td>单用户平均贡献</td></tr>
      <tr><td>ARPPU</td><td>总收入 ÷ 付费用户</td><td>付费用户价值深度</td></tr>
    </table>
  </div>

  <script>
    const chartOpts = {{ responsive: true, plugins: {{ legend: {{ display: false }} }} }};
    new Chart(document.getElementById('dauChart'), {{
      type: 'line',
      data: {{
        labels: {json.dumps(dau_labels)},
        datasets: [{{ data: {json.dumps(dau_values)}, borderColor: '#38bdf8', backgroundColor: 'rgba(56,189,248,0.15)', fill: true, tension: 0.3 }}]
      }},
      options: chartOpts
    }});
    new Chart(document.getElementById('newUserChart'), {{
      type: 'bar',
      data: {{
        labels: {json.dumps(new_labels)},
        datasets: [{{ data: {json.dumps(new_values)}, backgroundColor: '#a78bfa' }}]
      }},
      options: chartOpts
    }});
  </script>
</body>
</html>"""


def build_rfm_report(rfm: pd.DataFrame) -> str:
    seg = rfm.groupby("segment").agg(
        用户数=("user_id", "count"),
        平均购买次数=("buy_count", "mean"),
        平均最近活跃天数=("recency_days", "mean"),
    ).reset_index()
    seg["占比"] = (seg["用户数"] / seg["用户数"].sum() * 100).round(2)
    seg = seg.sort_values("用户数", ascending=False)

    strategies = {
        "重要价值用户": "VIP 专属权益、会员优先客服、新品首发邀请；维持高粘性，防止流失至竞品。",
        "重要发展用户": "满减券 + 复购激励（买二赠一）；推送关联品类，提升购买频次。",
        "重要保持用户": "定期唤醒（短信/Push）、限时秒杀；用积分体系拉回最近活跃度。",
        "重要挽留用户": "大额挽回券、专属客服回访；分析流失原因，个性化推荐历史偏好品类。",
        "一般价值用户": "日常运营触达，培养消费习惯；新人专享价引导二次购买。",
        "一般发展用户": "精准商品推荐 + 低门槛优惠券；引导完成首单或第二单。",
        "一般保持用户": "签到积分、内容营销（菜谱/攻略）；低打扰频次维护关系。",
        "一般挽留用户": "低成本自动化触达；对长期无行为用户可暂停营销节省成本。",
        "高意向待转化用户": "加购/收藏后 24h 内推送限时优惠券；购物车降价提醒，促进首单转化。",
        "活跃未付费用户": "新人首单立减、品类试用券；通过精准推荐降低决策门槛。",
        "沉睡用户": "低成本短信/邮件唤醒；超过 30 天无行为可暂停投放，控制获客成本。",
    }

    lines = [
        "# 用户分层画像及差异化运营策略建议",
        "",
        "## 一、RFM 模型说明",
        "",
        "| 维度 | 英文 | 含义 | 打分规则 |",
        "|------|------|------|----------|",
        "| R | Recency | 最近一次活跃距今天数 | 越近分数越高（1-5分） |",
        "| F | Frequency | 购买频次 | 越高分数越高 |",
        "| M | Monetary | 购买金额/次数 | 越高分数越高 |",
        "",
        f"> 分析基准日：{REFERENCE_DATE.date()}，共 {len(rfm):,} 名用户。",
        "",
        "## 二、用户分层分布",
        "",
        "| 用户分层 | 用户数 | 占比 | 平均购买次数 | 平均最近活跃(天) |",
        "|----------|--------|------|-------------|-----------------|",
    ]
    for _, row in seg.iterrows():
        lines.append(
            f"| {row['segment']} | {row['用户数']:,} | {row['占比']}% | "
            f"{row['平均购买次数']:.2f} | {row['平均最近活跃天数']:.1f} |"
        )

    lines += ["", "## 三、各分层用户画像", ""]
    portraits = {
        "重要价值用户": "近期活跃、高频购买、消费力强。是平台 GMV 核心贡献者。",
        "重要发展用户": "最近有活跃且客单不低，但购买频次尚低。具备成长为核心用户的潜力。",
        "重要保持用户": "历史消费高但最近活跃度下降，存在流失风险，需及时唤醒。",
        "重要挽留用户": "曾经高消费但长期未回归，属于高价值流失预警群体。",
        "一般价值用户": "活跃且有一定频次，但消费金额有限，可通过促销提升 ARPPU。",
        "一般发展用户": "近期活跃、购买次数少，处于培养期。",
        "一般保持用户": "有一定历史消费但近期冷淡，维护成本应控制在合理范围。",
        "一般挽留用户": "历史有消费但长期未回归，需挽回策略。",
        "高意向待转化用户": "有加购或收藏行为但未完成支付，处于转化漏斗最后一环，转化 ROI 最高。",
        "活跃未付费用户": "近期有浏览活跃但无购买记录，可通过首单优惠引导转化。",
        "沉睡用户": "长期未活跃且无消费，对短期 GMV 贡献低，宜低成本的批量唤醒。",
    }
    for name in seg["segment"]:
        if name in portraits:
            lines.append(f"### {name}")
            lines.append(f"{portraits[name]}")
            lines.append("")

    lines += ["## 四、差异化运营策略建议", ""]
    lines.append("| 用户分层 | 策略方向 | 具体措施 |")
    lines.append("|----------|----------|----------|")
    for name in seg["segment"]:
        lines.append(f"| {name} | 精准营销 | {strategies.get(name, '—')} |")

    lines += [
        "",
        "## 五、营销优先级矩阵",
        "",
        "```",
        "        高消费(M)",
        "          ↑",
        "  重要挽留 ← → 重要价值",
        "  重要保持 ← → 重要发展",
        "          ↓",
        "        低消费",
        "```",
        "",
        "**执行建议：**",
        "1. **第一优先级**：重要价值 + 重要挽留（高 M），投入 40% 营销预算。",
        "2. **第二优先级**：重要发展 + 重要保持，投入 35% 预算，聚焦提升 F 和 R。",
        "3. **第三优先级**：一般用户群，以自动化运营为主，投入 25% 预算。",
        "",
        f"![RFM分层分布](charts/rfm_segments.png)",
        "",
    ]
    return "\n".join(lines)


def build_value_report(metrics: dict, df: pd.DataFrame, cat: pd.DataFrame, rfm: pd.DataFrame) -> str:
    buyers = metrics["paying_uv"]
    non_buyers = metrics["uv"] - buyers
    buy_actions = metrics["buy"]
    avg_orders_per_buyer = buy_actions / buyers if buyers else 0

    cat_top = cat.head(15)
    cat_lines = []
    for _, row in cat_top.iterrows():
        cat_lines.append(
            f"| {int(row['category'])} | {int(row['pv']):,} | {int(row['buy']):,} | "
            f"{row['cvr']:.2%} | {int(row['uv']):,} |"
        )

    # 付费用户 vs 非付费用户行为对比
    user_beh = df.groupby(["user_id", "behavior"]).size().unstack(fill_value=0)
    user_beh["is_payer"] = user_beh.get("buy", 0) > 0
    payer_avg = user_beh[user_beh["is_payer"]][["pv", "cart", "fav", "buy"]].mean()
    non_payer_avg = user_beh[~user_beh["is_payer"]][["pv", "cart", "fav", "buy"]].mean()

    # 品类与付费关系：购买用户偏好品类
    buyer_cats = df[df["behavior"] == "buy"].groupby("category").size().sort_values(ascending=False).head(10)

    return f"""# 用户价值与付费行为分析报告

## 一、核心付费指标概览

| 指标 | 数值 | 说明 |
|------|------|------|
| 总 UV | {metrics['uv']:,} | 平台独立访客 |
| 付费 UV | {buyers:,} | 至少有一次购买行为 |
| 非付费 UV | {non_buyers:,} | 仅浏览/加购/收藏 |
| **付费率** | **{metrics['pay_rate']:.2%}** | 付费UV / 总UV |
| 购买次数 | {buy_actions:,} | 总成交笔数 |
| 人均购买次数（付费用户） | {avg_orders_per_buyer:.2f} | 复购水平 |
| **GMV（估算）** | **¥{metrics['gmv']:,.0f}** | 购买次数 × 假设客单价 ¥{ASSUMED_AOV:.0f} |
| **ARPU** | **¥{metrics['arpu']:.2f}** | GMV / 总UV |
| **ARPPU** | **¥{metrics['arppu']:.2f}** | GMV / 付费UV |

> 注：原始数据不含商品价格，GMV/ARPU/ARPPU 基于行业参考客单价 ¥{ASSUMED_AOV:.0f} 估算，用于横向对比与趋势分析。

---

## 二、付费率深度分析

### 2.1 漏斗视角

| 环节 | 转化率 |
|------|--------|
| PV → 加购 | {metrics['ctr_pv_to_cart']:.2%} |
| PV → 购买 | {metrics['cvr_pv_to_buy']:.2%} |
| UV → 购买（付费率） | {metrics['pay_rate']:.2%} |
| 加购 → 购买 | {metrics['cart_to_buy_cvr']:.2%} |

**洞察：**
- 整体付费率 {metrics['pay_rate']:.2%}，说明 **97% 以上用户尚未完成购买**，拉新与转化仍有巨大空间。
- 加购→购买转化率 {metrics['cart_to_buy_cvr']:.2%}，购物车放弃率较高，可通过限时优惠、运费减免促进下单。
- ARPPU（¥{metrics['arppu']:.1f}）显著高于 ARPU（¥{metrics['arpu']:.2f}），付费用户价值高度集中，应重点维护高价值付费人群。

### 2.2 付费 vs 非付费用户行为差异

| 行为类型 | 付费用户人均 | 非付费用户人均 |
|----------|-------------|---------------|
| 浏览(pv) | {payer_avg.get('pv', 0):.1f} | {non_payer_avg.get('pv', 0):.1f} |
| 加购(cart) | {payer_avg.get('cart', 0):.1f} | {non_payer_avg.get('cart', 0):.1f} |
| 收藏(fav) | {payer_avg.get('fav', 0):.1f} | {non_payer_avg.get('fav', 0):.1f} |
| 购买(buy) | {payer_avg.get('buy', 0):.1f} | {non_payer_avg.get('buy', 0):.1f} |

付费用户人均浏览次数低于非付费用户，但购买转化更果断，说明 **付费用户决策路径更短**；非付费用户浏览更深但缺乏转化，应针对高加购/收藏用户（高意向待转化群）精准推送优惠促成首单。

---

## 三、日活（DAU）分析

| 指标 | 数值 |
|------|------|
| 日均 DAU | {metrics['dau_mean']:,.0f} |
| DAU 峰值 | {metrics['dau_max']:,} |
| DAU / UV 比 | {metrics['dau_mean'] / metrics['uv'] * 27:.2%}（27天周期估算） |

DAU 是衡量用户粘性的核心指标。建议：
1. 建立 DAU 与 GMV 的关联监控，识别「活跃但不付费」用户群。
2. 对 DAU 下降日期进行归因（节假日、竞品活动、系统故障等）。
3. 通过签到、限时秒杀等手段提升 DAU，再经由漏斗转化为付费。

![DAU趋势](charts/dau_trend.png)

---

## 四、商品品类与用户行为关系

### 4.1 TOP15 品类流量与转化

| 品类ID | PV | 购买 | CVR | UV |
|--------|-----|------|-----|-----|
{chr(10).join(cat_lines)}

### 4.2 品类洞察

1. **高流量高转化品类**：CVR 高于均值（{cat['cvr'].mean():.2%}）的品类应加大曝光与库存，作为 GMV 主力。
2. **高流量低转化品类**：PV 高但 CVR 低，需优化详情页、定价或评价；同时是 **广告投放** 的重点（商家愿为曝光付费）。
3. **低流量高转化品类**：适合精准推荐与「猜你喜欢」，以长尾策略提升整体 CVR。

### 4.3 付费用户偏好品类 TOP10

| 品类ID | 购买次数 |
|--------|----------|
""" + "\n".join(f"| {int(k)} | {int(v):,} |" for k, v in buyer_cats.items()) + f"""

---

## 五、业务策略建议

| 方向 | 策略 | 预期效果 |
|------|------|----------|
| 提升付费率 | 加购用户 24h 内推送优惠券 | 提升加购→购买转化 |
| 提升 ARPPU | 满减凑单、关联推荐 | 提升客单价与复购 |
| 提升 ARPU | 扩大付费用户基数（新人首单） | 从 2.87% 付费率向 5%+ 迈进 |
| 品类运营 | 重点运营高 CVR 品类 | 优化 GMV 结构 |
| 会员体系 | 面向 RFM「重要价值」用户推会员 | 锁定高 LTV 用户 |

![品类分析](charts/category_top10.png)
![转化漏斗](charts/funnel.png)
"""




## 1. 数据加载与预览

In [ ]:
df = load_data()
print(f"数据行数: {len(df):,}")
print(f"用户数: {df['user_id'].nunique():,}")
print(f"时间范围: {df['datetime'].min()} ~ {df['datetime'].max()}")
print(f"\n行为分布:")
display(df['behavior'].value_counts())
df.head(10)

## 2. 流量漏斗与核心 KPI

In [ ]:
metrics = compute_funnel_metrics(df)
retention = compute_retention(df)

kpi_df = pd.DataFrame([
    {"指标": "PV", "数值": f"{metrics['pv']:,}", "说明": "页面浏览量"},
    {"指标": "UV", "数值": f"{metrics['uv']:,}", "说明": "独立访客"},
    {"指标": "DAU 日均", "数值": f"{metrics['dau_mean']:,.0f}", "说明": f"峰值 {metrics['dau_max']:,}"},
    {"指标": "付费率", "数值": f"{metrics['pay_rate']:.2%}", "说明": "付费UV/总UV"},
    {"指标": "CVR (PV→购买)", "数值": f"{metrics['cvr_pv_to_buy']:.2%}", "说明": "购买/PV"},
    {"指标": "GMV (估算)", "数值": f"¥{metrics['gmv']:,.0f}", "说明": f"客单价 ¥{ASSUMED_AOV:.0f}"},
    {"指标": "ARPU", "数值": f"¥{metrics['arpu']:.2f}", "说明": "GMV/总UV"},
    {"指标": "ARPPU", "数值": f"¥{metrics['arppu']:.2f}", "说明": "GMV/付费UV"},
    {"指标": "D1 留存均值", "数值": f"{retention['d1_retention_avg']:.2%}", "说明": "次日留存"},
    {"指标": "D7 留存均值", "数值": f"{retention['d7_retention_avg']:.2%}", "说明": "7日留存"},
])
display(kpi_df)

In [ ]:
# 行为转化漏斗图
stages = ["浏览(PV)", "加购", "收藏", "购买"]
values = [metrics["pv"], metrics["cart"], metrics["fav"], metrics["buy"]]
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#4C6EF5", "#228BE6", "#15AABF", "#12B886"]
bars = ax.barh(stages[::-1], values[::-1], color=colors[::-1])
for bar, val in zip(bars, values[::-1]):
    ax.text(bar.get_width() + max(values) * 0.01, bar.get_y() + bar.get_height() / 2,
            f"{val:,}", va="center", fontsize=10)
ax.set_xlabel("行为次数")
ax.set_title("用户行为转化漏斗")
plt.tight_layout()
plt.savefig(CHARTS_DIR / "funnel.png", dpi=150)
plt.show()

## 3. DAU 趋势与留存

In [ ]:
dau = pd.Series(metrics["dau_series"])
dau.index = pd.to_datetime(dau.index)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(dau.index, dau.values, marker="o", color="#4C6EF5", linewidth=2)
ax.fill_between(dau.index, dau.values, alpha=0.15, color="#4C6EF5")
ax.set_title("DAU 日活跃趋势")
ax.set_ylabel("DAU")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(CHARTS_DIR / "dau_trend.png", dpi=150)
plt.show()

## 4. RFM 用户分层

In [ ]:
rfm = build_rfm(df)
seg_summary = rfm.groupby("segment").agg(
    用户数=("user_id", "count"),
    平均购买次数=("buy_count", "mean"),
    平均最近活跃天数=("recency_days", "mean"),
).reset_index()
seg_summary["占比"] = (seg_summary["用户数"] / seg_summary["用户数"].sum() * 100).round(2)
seg_summary = seg_summary.sort_values("用户数", ascending=False)
display(seg_summary)

In [ ]:
seg_counts = rfm["segment"].value_counts()
fig, ax = plt.subplots(figsize=(9, 5))
seg_counts.plot(kind="bar", ax=ax, color="#7950F2")
ax.set_title("RFM 用户分层分布")
ax.set_ylabel("用户数")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(CHARTS_DIR / "rfm_segments.png", dpi=150)
plt.show()

In [ ]:
rfm_report = build_rfm_report(rfm)
display(Markdown(rfm_report.replace("charts/", "output/charts/")))

## 5. 品类分析

In [ ]:
cat = category_analysis(df)
display(cat.head(15))

In [ ]:
top10 = cat.head(10)
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(top10))
w = 0.35
ax.bar(x - w / 2, top10["pv"], w, label="PV", color="#74C0FC")
ax.bar(x + w / 2, top10["buy"], w, label="购买", color="#12B886")
ax.set_xticks(x)
ax.set_xticklabels(top10["category"].astype(str), rotation=45, ha="right")
ax.set_title("TOP10 品类流量与购买对比")
ax.legend()
plt.tight_layout()
plt.savefig(CHARTS_DIR / "category_top10.png", dpi=150)
plt.show()

## 6. 用户价值与付费行为分析报告

In [ ]:
value_report = build_value_report(metrics, df, cat, rfm)
display(Markdown(value_report.replace("charts/", "output/charts/")))

## 7. 交互式 HTML 看板

In [ ]:
html = build_html_dashboard(metrics, retention)
html_path = OUTPUT_DIR / "用户行为流量漏斗分析看板.html"
html_path.write_text(html, encoding="utf-8")
print(f"HTML 看板已保存: {html_path}")
display(HTML(html))

## 8. 导出结果文件

In [ ]:
rfm.to_csv(OUTPUT_DIR / "rfm_user_segments.csv", index=False)
cat.to_csv(OUTPUT_DIR / "category_analysis.csv", index=False)
(OUTPUT_DIR / "用户分层画像及差异化运营策略建议.md").write_text(rfm_report, encoding="utf-8")
(OUTPUT_DIR / "用户价值与付费行为分析报告.md").write_text(value_report, encoding="utf-8")

summary = {k: v for k, v in metrics.items() if not isinstance(v, dict)}
summary.update(retention)
(OUTPUT_DIR / "metrics_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("=== 分析完成，已导出 ===")
print(f"PV: {metrics['pv']:,} | UV: {metrics['uv']:,} | 付费率: {metrics['pay_rate']:.2%}")
print(f"GMV(估): ¥{metrics['gmv']:,.0f} | ARPU: ¥{metrics['arpu']:.2f} | ARPPU: ¥{metrics['arppu']:.2f}")
print(f"产出目录: {OUTPUT_DIR}")